# 추가 백엔드-5: Mastersample(마스터 샘플) 완료 여부 스캔

첨부 사양(2026-01-29 추가 백엔드-5 설계)에 따라,
- KST(Asia/Seoul) 기준으로 주/야간을 정의하고
- 선택한 **prod_day(YYYYMMDD)** 와 **shift_type(day/night)** 에 대해
- `d1_machine_log."Main_machine_log"` 에서 Mastersample 완료 문구(`Mastersample All OK`) 존재 여부를 스캔하여
- 아래 형태의 DataFrame을 생성합니다.

컬럼:
- `prod_day` (text, YYYYMMDD)
- `shift_type` (day / night)
- `Mastersample` (O / X)
- `updated_at` (DataFrame 생성/업데이트 시각)


In [1]:
# (선택) 최초 1회만 필요
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

In [11]:
from __future__ import annotations

import time
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from typing import Iterable, List, Literal, Optional, Tuple

import pandas as pd
from sqlalchemy import create_engine, text, bindparam
from sqlalchemy.engine import Engine

# =========================
# 1) DB 설정 (사양 그대로)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

KST = "Asia/Seoul"

# =========================
# 2) 상수/시간대 정의
# =========================
ShiftType = Literal["day", "night"]

# ✅ Mastersample 스캔 전용(08:20~09:00 / 20:20~21:00) 조건 제거
#    -> Mastersample 검색도 주/야간 생산 구간(SHIFT_WINDOW) 전체를 그대로 사용

# 주/야간 생산 구간(겹치지 않도록)
# - day   : [D] 08:30:00 ~ 20:29:59 (포함)
# - night : [D] 20:30:00 ~ 23:59:59 + [D+1] 00:00:00 ~ 08:29:59 (포함)
SHIFT_WINDOW = {
    "day":   ("08:30:00", "20:29:59"),
    "night": ("20:30:00", "08:29:59"),  # night는 D~D+1로 이어짐(아래 helper에서 처리)
}

TABLE_FQN = 'd1_machine_log."Main_machine_log"'  # 대문자 M 주의 (따옴표 필요)

In [12]:
def make_engine(pool_size: int = 1, max_overflow: int = 0) -> Engine:
    """백엔드 상시 연결 1개 수준으로 최소화."""
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=pool_size,
        max_overflow=max_overflow,
        pool_pre_ping=True,   # 끊김 감지
        pool_recycle=1800,    # 장시간 idle 대비
    )


def connect_with_retry(sleep_sec: int = 2) -> Engine:
    """DB 접속 실패 시 재시도(무한)."""
    while True:
        try:
            eng = make_engine()
            with eng.connect() as conn:
                conn.execute(text("SELECT 1"))
            print("[INFO] DB connected")
            return eng
        except Exception as e:
            print(f"[WARN] DB connect failed: {e}  -> retry in {sleep_sec}s")
            time.sleep(sleep_sec)


engine = connect_with_retry()


[INFO] DB connected


In [13]:
def _parse_prod_day(prod_day: str) -> date:
    """YYYYMMDD -> date"""
    return datetime.strptime(prod_day, "%Y%m%d").date()


def _kst_now() -> pd.Timestamp:
    return pd.Timestamp.now(tz=KST)


def shift_window_kst(prod_day: str, shift_type: ShiftType) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """사양의 주/야간 생산 구간을 KST 기준 start/end로 반환."""
    d = _parse_prod_day(prod_day)
    if shift_type == "day":
        start_s, end_s = SHIFT_WINDOW["day"]
        start = pd.Timestamp(f"{d.isoformat()} {start_s}", tz=KST)
        end   = pd.Timestamp(f"{d.isoformat()} {end_s}", tz=KST)
        return start, end

    # night
    start_s, end_s = SHIFT_WINDOW["night"]  # end는 다음날 08:29:59
    start = pd.Timestamp(f"{d.isoformat()} {start_s}", tz=KST)
    end   = pd.Timestamp(f"{(d + timedelta(days=1)).isoformat()} {end_s}", tz=KST)
    return start, end


def mastersample_scan_window(prod_day: str, shift_type: ShiftType) -> Tuple[str, str]:
    """사양의 Mastersample 스캔 시간창 (시간만). inclusive 비교에 사용."""
    return MASTER_SAMPLE_WINDOW[shift_type]


def check_mastersample(prod_day: str, shift_type: ShiftType, eng: Engine) -> Literal["O", "X"]:
    # 1) KST 기준 shift window -> DB 비교용 naive timestamp로 변환
    start_ts_kst, end_ts_kst = shift_window_kst(prod_day, shift_type)

    # DB(end_day/end_time)이 "현지시간(KST)" 기준으로 저장돼 있다고 가정하고 tz 제거(naive)
    start_ts = start_ts_kst.tz_convert(KST).tz_localize(None)
    end_ts   = end_ts_kst.tz_convert(KST).tz_localize(None)

    # 2) window가 걸치는 날짜(= end_day 후보) 목록 생성 (주간이면 1일, 야간이면 보통 2일)
    d0 = start_ts.date()
    d1 = end_ts.date()
    end_days = []
    cur = d0
    while cur <= d1:
        end_days.append(cur.strftime("%Y%m%d"))
        cur += timedelta(days=1)

    # 3) end_ts(timestamp)로 범위 필터 + Mastersample 문구 스캔
    sql = (
        text("""
            WITH base AS (
                SELECT
                    end_day,
                    split_part(end_time, '.', 1) AS end_time_hms,
                    contents
                FROM d1_machine_log."Main_machine_log"
                WHERE end_day IN :end_days
            ),
            ts AS (
                SELECT
                    to_timestamp(end_day || ' ' || end_time_hms, 'YYYYMMDD HH24:MI:SS') AS end_ts,
                    contents
                FROM base
            )
            SELECT 1
            FROM ts
            WHERE end_ts >= :start_ts
              AND end_ts <= :end_ts
              AND contents ILIKE '%Mastersample All OK%'
            LIMIT 1
        """)
        .bindparams(bindparam("end_days", expanding=True))
    )

    with eng.connect() as conn:
        row = conn.execute(
            sql,
            {
                "end_days": end_days,
                "start_ts": start_ts,
                "end_ts": end_ts,
            },
        ).first()

    return "O" if row is not None else "X"


def build_mastersample_df(
    prod_days: Iterable[str],
    shift_type: ShiftType,
    eng: Engine,
) -> pd.DataFrame:
    """요청 포맷대로 DataFrame 생성."""
    updated_at = _kst_now()
    rows = []
    for pday in prod_days:
        flag = check_mastersample(pday, shift_type, eng)
        rows.append(
            {
                "prod_day": pday,
                "shift_type": shift_type,
                "Mastersample": flag,
                "updated_at": updated_at,
            }
        )
    return pd.DataFrame(rows)


In [14]:
# =========================
# 예시 실행
# =========================
# 1) 단일 prod_day
prod_day = "20260127"
shift_type: ShiftType = "day"  # "night" 로 바꾸면 야간 스캔

df1 = build_mastersample_df([prod_day], shift_type, engine)
df1


,prod_day,shift_type,Mastersample,updated_at
0,20260127,day,X,2026-01-29 19:06:49.505022+09:00


In [15]:
# 2) 날짜 범위 (DATE_FROM~DATE_TO)
DATE_FROM = "20260120"
DATE_TO   = "20260129"  # 포함

start_d = _parse_prod_day(DATE_FROM)
end_d   = _parse_prod_day(DATE_TO)
prod_days = [(start_d + timedelta(days=i)).strftime("%Y%m%d") for i in range((end_d - start_d).days + 1)]

shift_type = "day"
df2 = build_mastersample_df(prod_days, shift_type, engine)

df2


,prod_day,shift_type,Mastersample,updated_at
0,20260120,day,X,2026-01-29 19:06:54.922027+09:00
1,20260121,day,X,2026-01-29 19:06:54.922027+09:00
2,20260122,day,X,2026-01-29 19:06:54.922027+09:00
3,20260123,day,X,2026-01-29 19:06:54.922027+09:00
4,20260124,day,X,2026-01-29 19:06:54.922027+09:00
5,20260125,day,X,2026-01-29 19:06:54.922027+09:00
6,20260126,day,X,2026-01-29 19:06:54.922027+09:00
7,20260127,day,X,2026-01-29 19:06:54.922027+09:00
8,20260128,day,X,2026-01-29 19:06:54.922027+09:00
9,20260129,day,X,2026-01-29 19:06:54.922027+09:00
